In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Load engineered dataset
df = pd.read_csv('../thesis_dataset/engineered_dataset.csv')
print(f"Loaded: {df.shape}")

# Feature Sets
# Model A — Performance only
# Rationale: Rate stats (per90) capture efficiency, raw totals
# capture volume. Age and age_squared model the non-linear
# career value curve observed in EDA.
features_A = [
    'goals_per90',
    'assists_per90',
    'goal_contributions',
    'minutes_played',
    'age',
    'age_squared'
]

# Model B — Performance + Contextual
# Rationale: Club prestige, position, and foot added to test
# whether contextual factors improve predictive accuracy
# beyond performance metrics alone.
features_B = features_A + [
    'club_total_value',
    'pos_Attack',
    'pos_Defender',
    'pos_Midfield',
    'foot_left',
    'foot_right'
]

# Model C — Same features as B, non-linear algorithm
# Rationale: Random Forest captures non-linear interactions
# between features that Ridge Regression cannot model.
features_C = features_B

# Temporal Train/Test Split
# Test set: 2023 and 2024 seasons (most recent, unseen data)
# Train set: all prior seasons (2012–2022)
TEST_SEASONS = [2023, 2024]

def temporal_split(df, features, target='log_market_value'):
    model_df = df[features + [target, 'season']].dropna()
    train = model_df[~model_df['season'].isin(TEST_SEASONS)]
    test  = model_df[ model_df['season'].isin(TEST_SEASONS)]
    X_train = train[features]
    X_test  = test[features]
    y_train = train[target]
    y_test  = test[target]
    return X_train, X_test, y_train, y_test

X_train_A, X_test_A, y_train_A, y_test_A = temporal_split(df, features_A)
X_train_B, X_test_B, y_train_B, y_test_B = temporal_split(df, features_B)
X_train_C, X_test_C, y_train_C, y_test_C = temporal_split(df, features_C)

print(f"\nTrain size: {X_train_B.shape[0]}")
print(f"Test size:  {X_test_B.shape[0]}")

# Model A: Ridge Regression — Performance Only
pipeline_A = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])

pipeline_A.fit(X_train_A, y_train_A)
y_pred_A = pipeline_A.predict(X_test_A)

r2_A   = r2_score(y_test_A, y_pred_A)
rmse_A = np.sqrt(mean_squared_error(y_test_A, y_pred_A))

print(f"\n── Model A: Ridge — Performance Only ──")
print(f"R²:   {r2_A:.4f}")
print(f"RMSE: {rmse_A:.4f}")

# Model B: Ridge Regression — Performance + Context
pipeline_B = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])

pipeline_B.fit(X_train_B, y_train_B)
y_pred_B = pipeline_B.predict(X_test_B)

r2_B   = r2_score(y_test_B, y_pred_B)
rmse_B = np.sqrt(mean_squared_error(y_test_B, y_pred_B))

print(f"\n── Model B: Ridge — Performance + Context ──")
print(f"R²:   {r2_B:.4f}")
print(f"RMSE: {rmse_B:.4f}")

# Model C: Random Forest — Performance + Context
pipeline_C = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_C.fit(X_train_C, y_train_C)
y_pred_C = pipeline_C.predict(X_test_C)

r2_C   = r2_score(y_test_C, y_pred_C)
rmse_C = np.sqrt(mean_squared_error(y_test_C, y_pred_C))

print(f"\n── Model C: Random Forest — Performance + Context ──")
print(f"R²:   {r2_C:.4f}")
print(f"RMSE: {rmse_C:.4f}")

# Results Summary
print("\n\n══════════════════════════════════════════")
print("  MODEL COMPARISON SUMMARY")
print("══════════════════════════════════════════")
print(f"{'Model':<12} {'Algorithm':<20} {'Features':<25} {'R²':>6} {'RMSE':>7}")
print(f"{'─'*12} {'─'*20} {'─'*25} {'─'*6} {'─'*7}")
print(f"{'A':<12} {'Ridge':<20} {'Performance only':<25} {r2_A:>6.4f} {rmse_A:>7.4f}")
print(f"{'B':<12} {'Ridge':<20} {'Performance + Context':<25} {r2_B:>6.4f} {rmse_B:>7.4f}")
print(f"{'C':<12} {'Random Forest':<20} {'Performance + Context':<25} {r2_C:>6.4f} {rmse_C:>7.4f}")

# Save pipelines
import joblib
joblib.dump(pipeline_A, '../thesis_dataset/model_A.pkl')
joblib.dump(pipeline_B, '../thesis_dataset/model_B.pkl')
joblib.dump(pipeline_C, '../thesis_dataset/model_C.pkl')
print("\nModels saved to thesis_dataset/")

Loaded: (3819, 28)

Train size: 3249
Test size:  568

── Model A: Ridge — Performance Only ──
R²:   0.1582
RMSE: 0.9517

── Model B: Ridge — Performance + Context ──
R²:   0.4787
RMSE: 0.7489

── Model C: Random Forest — Performance + Context ──
R²:   0.5887
RMSE: 0.6652


══════════════════════════════════════════
  MODEL COMPARISON SUMMARY
══════════════════════════════════════════
Model        Algorithm            Features                      R²    RMSE
──────────── ──────────────────── ───────────────────────── ────── ───────
A            Ridge                Performance only          0.1582  0.9517
B            Ridge                Performance + Context     0.4787  0.7489
C            Random Forest        Performance + Context     0.5887  0.6652

Models saved to thesis_dataset/
